# FIFA World Cup 2026 — Player Performance EDA

**Author:** Rushikesh Waje  
**Dataset:** 54,600 match records · 1,245 players · 48 teams · 75 features  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## Objectives
1. Understand dataset structure and quality
2. Identify top scorers, assisters, and POTM winners
3. Analyze xG vs actual goals — who overperformed?
4. Explore clutch performance by tournament stage
5. Correlate market value with match rating
6. Compare physical metrics across positions
7. Rank teams by goals and average rating

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor'  : '#111827',
    'axes.edgecolor'  : '#1e2d42',
    'axes.labelcolor' : '#94a3b8',
    'text.color'      : '#e2e8f0',
    'xtick.color'     : '#64748b',
    'ytick.color'     : '#64748b',
    'grid.color'      : '#1e2d42',
    'grid.alpha'      : 0.5,
    'font.family'     : 'DejaVu Sans',
})

PALETTE = ['#00d4ff', '#ffd700', '#00e676', '#b388ff', '#ff9800', '#ff5252']
POS_COLORS = {'Forward':'#ff5252','Midfielder':'#00d4ff','Defender':'#00e676','Goalkeeper':'#ff9800'}

print('Libraries loaded ✅')

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('../data/fifa_world_cup_2026_player_performance.csv')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

In [ ]:
print('Unique players :', df['player_name'].nunique())
print('Unique teams   :', df['team'].nunique())
print('Unique matches :', df['match_id'].nunique())
print('Tournament stages:', df['tournament_stage'].unique())
print('Positions        :', df['position'].unique())
print('Date range       :', df['match_date'].min(), '→', df['match_date'].max())

In [ ]:
df[['goals','assists','player_rating','performance_score',
    'distance_covered_km','market_value_eur']].describe().round(2)

## 2. Top Scorers — Golden Boot Race

In [ ]:
top_scorers = (
    df.groupby('player_name')
    .agg(goals=('total_goals_tournament','max'), team=('team','first'))
    .reset_index()
    .sort_values('goals', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ffd700','#c0c0c0','#cd7f32'] + ['#ff5252aa']*7
bars = ax.barh(top_scorers['player_name'][::-1], top_scorers['goals'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.4, height=0.6)
for bar, val in zip(bars, top_scorers['goals'][::-1]):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
            f'{val} ⚽', va='center', fontsize=9, color='#e2e8f0')
ax.set_title('Top 10 Goal Scorers — FIFA World Cup 2026', fontsize=13, fontweight='bold',
             color='#e2e8f0', pad=12)
ax.set_xlabel('Goals Scored', color='#94a3b8')
ax.set_xlim(0, 13)
ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/top_scorers.png', dpi=150, bbox_inches='tight')
plt.show()
print(top_scorers.to_string(index=False))

## 3. xG vs Actual Goals — Overperformers

In [ ]:
xg_df = (
    df.groupby('player_name')
    .agg(actual=('goals','sum'), xg=('expected_goals_xg','sum'))
    .reset_index()
)
xg_df = xg_df[xg_df['actual'] >= 8].sort_values('actual', ascending=False).head(12)
xg_df['overperf'] = (xg_df['actual'] - xg_df['xg']).round(1)

x = range(len(xg_df))
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i-0.2 for i in x], xg_df['actual'], width=0.38, color='#00e676aa',
       edgecolor='#00e676', label='Actual Goals')
ax.bar([i+0.2 for i in x], xg_df['xg'], width=0.38, color='#ff9800aa',
       edgecolor='#ff9800', label='Expected Goals (xG)')
ax.set_xticks(list(x))
ax.set_xticklabels([n.split()[-1] for n in xg_df['player_name']], rotation=30, ha='right')
ax.set_title('xG vs Actual Goals — Who Overperformed Their Model?',
             fontsize=12, fontweight='bold', color='#e2e8f0', pad=12)
ax.legend(framealpha=0.2, labelcolor='white')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/xg_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clutch Performance Score by Tournament Stage

In [ ]:
stage_order = ['Group Stage','Round of 32','Round of 16',
               'Quarter Finals','Semi Finals','Third Place Match','Final']
clutch = (
    df.groupby('tournament_stage')['clutch_performance_score']
    .mean().round(2).reset_index()
)
clutch['tournament_stage'] = pd.Categorical(clutch['tournament_stage'],
                                             categories=stage_order, ordered=True)
clutch = clutch.sort_values('tournament_stage')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(clutch['tournament_stage'], clutch['clutch_performance_score'],
        marker='o', color='#ffd700', linewidth=2.5, markersize=8, markerfacecolor='#ffd700')
ax.fill_between(clutch['tournament_stage'], clutch['clutch_performance_score'],
                alpha=0.15, color='#ffd700')
for i, row in clutch.iterrows():
    ax.annotate(f"{row['clutch_performance_score']}",
                (row['tournament_stage'], row['clutch_performance_score']),
                textcoords='offset points', xytext=(0,8),
                ha='center', fontsize=9, color='#ffd700')
ax.set_title('Clutch Performance Score by Tournament Stage',
             fontsize=12, fontweight='bold', color='#e2e8f0', pad=12)
ax.set_ylim(52, 62)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/clutch_by_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Market Value vs Player Rating (Correlation)

In [ ]:
corr_df = (
    df[df['player_rating'] > 0]
    .groupby('player_name')
    .agg(mv=('market_value_eur','first'), rating=('player_rating','mean'),
         position=('position','first'))
    .reset_index()
    .sample(400, random_state=42)
)
corr_val = corr_df[['mv','rating']].corr().iloc[0,1].round(3)

fig, ax = plt.subplots(figsize=(9,5))
for pos, grp in corr_df.groupby('position'):
    ax.scatter(grp['mv']/1e6, grp['rating'], label=pos,
               color=POS_COLORS[pos], alpha=0.7, s=30, edgecolors='none')
ax.set_xlabel('Market Value (€M)', color='#94a3b8')
ax.set_ylabel('Avg Match Rating', color='#94a3b8')
ax.set_title(f'Market Value vs Match Rating   |   Pearson r = {corr_val}',
             fontsize=12, fontweight='bold', color='#e2e8f0', pad=12)
ax.legend(framealpha=0.2, labelcolor='white')
ax.grid(alpha=0.2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/market_value_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation: {corr_val} — moderate positive relationship')

## 6. Physical Metrics by Position

In [ ]:
phys = (
    df.groupby('position')
    .agg(distance=('distance_covered_km','mean'),
         speed=('top_speed_kmh','mean'),
         stamina=('stamina_score','mean'))
    .reset_index().round(2)
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [('distance','Distance Covered (km)'),
           ('speed','Top Speed (km/h)'),
           ('stamina','Stamina Score')]
for ax, (col, label) in zip(axes, metrics):
    colors = [POS_COLORS[p] for p in phys['position']]
    ax.bar(phys['position'], phys[col], color=[c+'aa' for c in colors],
           edgecolor=colors, linewidth=1.2, width=0.5)
    for i, v in enumerate(phys[col]):
        ax.text(i, v+0.1, str(v), ha='center', fontsize=9, color='#e2e8f0')
    ax.set_title(label, fontsize=10, color='#e2e8f0')
    ax.tick_params(axis='x', rotation=15)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.2)

fig.suptitle('Physical Performance Metrics by Position',
             fontsize=13, fontweight='bold', color='#e2e8f0', y=1.02)
plt.tight_layout()
plt.savefig('../assets/physical_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Team Comparison — Goals & Ratings

In [ ]:
team_g = df.groupby('team')['goals'].sum().sort_values(ascending=False).head(10).reset_index()
country_r = df.groupby('team')['tournament_rating'].mean().sort_values(ascending=False).head(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bar_colors = ['#ffd700','#c0c0c0','#cd7f32'] + ['#b388ffaa']*7
axes[0].bar(team_g['team'], team_g['goals'], color=bar_colors,
            edgecolor='white', linewidth=0.4, width=0.6)
axes[0].set_title('Top 10 Teams — Total Goals', fontsize=11, color='#e2e8f0')
axes[0].tick_params(axis='x', rotation=35)
axes[0].spines[['top','right']].set_visible(False)
axes[0].grid(axis='y', alpha=0.2)

axes[1].barh(country_r['team'][::-1], country_r['tournament_rating'][::-1],
             color='#00d4ffaa', edgecolor='#00d4ff', linewidth=0.8, height=0.6)
axes[1].set_title('Top 10 Nations — Avg Tournament Rating', fontsize=11, color='#e2e8f0')
axes[1].set_xlim(3.5, 4.0)
axes[1].spines[['top','right','left','bottom']].set_visible(False)
axes[1].grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig('../assets/team_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Insights Summary

In [ ]:
print('=' * 60)
print('  FIFA WORLD CUP 2026 — KEY ANALYTICAL INSIGHTS')
print('=' * 60)

print(f"""
1. DATASET OVERVIEW
   • {df['player_name'].nunique():,} players across {df['team'].nunique()} nations
   • {df['match_id'].nunique():,} matches from Group Stage to Final
   • {df.shape[0]:,} total match records | Zero missing values

2. GOLDEN BOOT
   • Declan Saka (England) & Stuart McTominay (Scotland) — 10 goals each
   • Assist leader: Michail McCarey (Jamaica) — 8 assists

3. xG ANALYSIS
   • Memphis Zerrouki: 24 goals vs 3.22 xG — exceptional finisher
   • Several top scorers massively outperformed statistical expectations

4. CLUTCH FACTOR
   • Avg clutch score rises from 55.0 (Group Stage) to 59.4 (Final)
   • +8% increase shows elite players raise their game when it matters

5. MARKET VALUE vs PERFORMANCE
   • Pearson r = 0.41 — moderate positive correlation
   • Several low-value players outperformed expensive stars
   • Investment ≠ guaranteed performance at World Cup level

6. PHYSICAL INSIGHTS
   • Midfielders cover most ground: avg 4.50 km per match
   • Forwards edge marginally higher top speed: 19.85 km/h
   • Goalkeepers: lowest distance (2.41 km) and stamina (78.8)

7. TEAM PERFORMANCE
   • Qatar (host): topped goals chart with 95 goals
   • Japan: highest avg nation rating at 3.78
   • Netherlands close behind with 94 goals & strong ratings
""")
print('=' * 60)